<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Assignment 2: RAG
  </span>
</h2>

<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Student Name: Yaron Winter
  </span>
</h2>

In [1]:
print("Install required packages:")
!pip install --upgrade pip

!pip install -q google
#!pip install -q google-colab
!pip install -q google-api-python-client
print("Done Install!")

Install required packages:
Done Install!


<h2 style="text-align: center;">
  <span style="display: inline-block; border-bottom: 2px solid black; padding-bottom: 5px;">
    Task 1 - Naive Generation
  </span>
</h2>

In [2]:
import pandas as pd

TABLES_FOLDER = "/home/yaron/WorkEnv/data/nebius/weeks/week3/rag_and_ai/task/"
SORTED_TOP_10_TABLE = "top_10_questions.csv"
SORTED_FULL_TABLE = "sorted_by_id_benchmark.csv"

DISPLAY_FIELDS = ["financebench_id", "doc_name", "question_type", "question", "answer", "evidence"]

top_10_df = pd.read_csv(TABLES_FOLDER + SORTED_TOP_10_TABLE, index_col=0)
full_df = pd.read_csv(TABLES_FOLDER + SORTED_FULL_TABLE, index_col=0)

In [3]:
print("Tables (after removing metrics-generated + sorting by ID):")
print(f"len(top_10)={len(top_10_df)}, len(full)={len(full_df)}\n")
top_10_df[DISPLAY_FIELDS].head(5)

Tables (after removing metrics-generated + sorting by ID):
len(top_10)=10, len(full)=100



,financebench_id,doc_name,question_type,question,answer,evidence
0,financebench_id_00005,CORNING_2022_10K,domain-relevant,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,[{'evidence_text': 'Consolidated Balance Sheet...
1,financebench_id_00070,AMERICANWATERWORKS_2022_10K,domain-relevant,Does American Water Works have positive workin...,"No, American Water Works had negative working ...",[{'evidence_text': 'American Water Works Compa...
2,financebench_id_00080,PAYPAL_2022_10K,domain-relevant,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...,"[{'evidence_text': 'PayPal Holdings, Inc.\nCON..."
3,financebench_id_00206,JPMORGAN_2022_10K,domain-relevant,Are JPM's gross margins historically consisten...,"Since JPM is a financial institution, gross ma...",[{'evidence_text': 'Overview\nJPMorgan Chase &...
4,financebench_id_00215,VERIZON_2022_10K,domain-relevant,Is Verizon a capital intensive business based ...,Yes. Verizon's capital intensity ratio was app...,[{'evidence_text': 'Consolidated Balance Sheet...


In [4]:
# Get Nebius API Key
from getpass import getpass
#from google.colab import userdata
import os

def get_api_key(name="NEBIUS_API_KEY"):
    api_key = None
    try:
        api_key = os.environ[name]
        print("API Key is taken from the environmental parameters")
    except:
        try:
            api_key = userdata.get(name)
            print("API Key is taken from the google colab user data")
        except:
            try:
                api_key = getpass("Enter API key: ")
                print("API Key is taken from getpass")
            except:
                raise Exception("API Key for NEBIUS_API_KEY could not be found")

    assert api_key is not None, "API Key is None"
    return api_key

In [5]:
# Allocate AI client
from openai import OpenAI

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key = get_api_key(),
)

API Key is taken from the environmental parameters


In [6]:
# Define the LLM response generation code.
import json
def generate_response(attributes: dict,
                      model_name: str,
                      system_prompt: str,
                      user_prompt: str,
                      response_format: dict) -> dict:
    return client.chat.completions.create(
        model=model_name,
        response_format=response_format,
        messages=[
            {"role": "system", "content": system_prompt},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": f"{user_prompt} {json.dumps(attributes)}```"}
                ],
            },
        ]
    ).to_dict()
print("Done!")

Done!


In [7]:
# Define the prompts for the naive answering generation.
NAIVE_SYSTEM_PROMPT = """
You are an AI agent.
Your task is to answer the questions given to you.
The question to be answered will be presented by a simple JSON-formatted string,
as given in this example:

***Example:***
"{
'Question':'Does Corning have positive working capital based on FY2022 data?...',
}"

***Output Format:***
Please retrive just the generated description, according to the provided format.
"""

NAIVE_USER_PROMPT = """
Please answer this given question.
The question is provided below as JSON-Formatted string between triple backticks:

```json
"""

NAIVE_RESPONSE_SCHEMA = {
  "name": "generated_answer",
  "schema": {
    "type": "object",
    "properties": {
      "generated_answer": {
        "type": "string",
        "description": "An answer to the given question. The length of the answer should not be longer than 90 words"
      },
    },
    "required":["generated_answer"],
    "additionalProperties": False
  },
  "strict": True
}

NAIVE_RESPONSE_FORMAT = {"type": "json_schema", "json_schema": NAIVE_RESPONSE_SCHEMA}

In [15]:
# The code for the naive answers generation.
QUESTION = "Question"

def generate_naive_answer(question: str, model_name: str) -> str:
    q = {QUESTION: question}
    response = generate_response(model_name=model_name,
                                 attributes=q,
                                 user_prompt=NAIVE_USER_PROMPT,
                                 system_prompt=NAIVE_SYSTEM_PROMPT,
                                 response_format=NAIVE_RESPONSE_FORMAT)
    j = json.loads(response["choices"][0]["message"]["content"])
    return j["generated_answer"]
print("Done!")

Done!


In [16]:
# Generate naive answers to the top 10 entries.
naive_df = top_10_df[["financebench_id", "question_type", "question"]].copy()
naive_df["ground_truth"] = top_10_df.answer
naive_df["naive_answer"] = naive_df.question.apply(lambda x: generate_naive_answer(x, model_name="meta-llama/Llama-3.3-70B-Instruct"))
naive_df["verdict"] = "DEFAULT"
naive_df.head(10)

,financebench_id,question_type,question,ground_truth,naive_answer,verdict
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,Corning's FY2022 data shows total current asse...,DEFAULT
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,"No, American Water Works had negative working ...","According to FY2022 data, American Water Works...",DEFAULT
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...,"According to FY2022 data, Paypal's current ass...",DEFAULT
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,"Since JPM is a financial institution, gross ma...","JPM's gross margins are not a relevant metric,...",DEFAULT
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,Yes. Verizon's capital intensity ratio was app...,Verizon is considered a capital-intensive busi...,DEFAULT
5,financebench_id_00283,novel-generated,How much does Pfizer expect to pay to spin off...,77.78,Pfizer expects to pay approximately $12 billio...,DEFAULT
6,financebench_id_00288,novel-generated,Was there any drop in Cash & Cash equivalents ...,"Yes, there was a decline of ~42% between FY202...",No data is provided to determine if there was ...,DEFAULT
7,financebench_id_00299,novel-generated,Which of JPM's business segments had the lowes...,Corporate. Its net revenue was -$473 million.,Corporate segment had the lowest net revenue i...,DEFAULT
8,financebench_id_00302,novel-generated,Did Pfizer grow its PPNE between FY20 and FY21?,"Yes, change in PPNE was positive year over year","According to Pfizer's financial reports, its P...",DEFAULT
9,financebench_id_00382,novel-generated,Which region had the Highest EBITDAR Contribut...,Las Vegas resorts contributed ~90% of company ...,Las Vegas Strip had the highest EBITDAR contri...,DEFAULT


In [18]:
naive_df.to_csv("assignment2_naive_generation.csv")